In [1]:
import torch

# Câu lệnh tự động chọn thiết bị
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Đang sử dụng thiết bị: {device}")


Đang sử dụng thiết bị: cpu


# Notebook 00 — Master ingestion & batch assignment (Colab drive-shadow + local-temp standardize + HF raw ingest)

Workflow đúng của notebook này:

```text
Google Drive folder của BTC / nguồn tổ chức
→ drive-shadow copy sang folder Drive của team/bạn
→ remount Drive và kiểm tra Colab nhìn thấy đủ file
→ standardize-archives bằng local temp `/content/temp_extract`
→ tạo standardized raw layout trên Drive:
   raw_videos/
   metadata/
   standardize_archives_report.json
   standardize_progress.jsonl
   missing_metadata.json
   unmatched_metadata.json
→ upload standardized raw_videos + metadata lên Hugging Face raw repo theo version prefix
→ kiểm tra HF raw repo
→ ingest từ HF raw repo để tạo processed artifacts local
→ assign-batches tạo batch_manifest.csv + batch_*.txt
→ copy các audit reports từ standardize/drive-shadow vào processed release manifests
→ upload processed artifacts nhỏ lên Hugging Face processed repo
→ kiểm tra HF processed repo
```

Thiết kế repo:

```text
AIC2026_raw       = versioned canonical raw store
AIC2026_processed = release/control plane
```

Raw repo dùng version prefix:

```text
AIC2026_raw/
└── canonical_dataset_v002/
    ├── raw_videos/
    ├── metadata/
    └── manifests/
        ├── canonical_file_manifest.jsonl
        └── canonical_import_report.json
```

Processed repo dùng release prefix:

```text
AIC2026_processed/
└── canonical_release_v002/
    ├── tables/videos.parquet
    ├── raw_mapping/media_store_manifest.parquet
    └── manifests/
        ├── dataset_report.json
        ├── ingestion_errors.jsonl
        ├── missing_metadata.json
        ├── unmatched_metadata.json
        ├── batch_manifest.csv
        └── batch_*.txt
```

Điểm quan trọng:

- `drive_target_id` phải là **folder ID đúng của folder local `archive_source_dir`**. Ví dụ nếu `archive_source_dir = /content/drive/MyDrive/AIC2026/raw_dataset`, thì `drive_target_id` phải chính là folder ID của `raw_dataset` trên Google Drive.
- `temp_extract_dir` **phải nằm trên local runtime** để tránh ghi temp qua Google DriveFS. Không đặt temp trong `/content/drive`.
- `missing_metadata.json` và `unmatched_metadata.json` là audit reports của bước standardize/pairing raw video ↔ metadata. HF ingest không nên tự recompute hai file này bằng cách tải lại raw videos.
- HF ingest hiện tại là nhánh `ingest` dùng `--canonical-hf-repo-id` và `--canonical-hf-prefix`.


In [28]:
import os
import sys
from pathlib import Path
from dataclasses import dataclass

@dataclass
class WorkflowConfig:
    # 1. Hugging Face repos
    # Raw repo: versioned canonical raw store.
    hf_canonical_repo: str = "1thesudden/AIC2026_raw"
    raw_import_id: str = "canonical_dataset_v002"

    # Processed repo: release/control plane.
    hf_release_repo: str = "1thesudden/AIC2026_processed"
    release_id: str = "canonical_release_v002"

    # Upload standardized raw lên HF raw repo.
    # Bật True khi muốn chạy workflow HF raw → HF ingest.
    run_upload_standardized_raw: bool = True

    # 2. Google Drive IDs
    # Workflow bắt buộc: copy source Drive folder của BTC sang folder Drive của team/bạn trước.
    drive_source_id: str = "1o4Rrfmu5ZHhpt_y5UjbzPZSzJsHO2klK" # https://drive.google.com/drive/folders/1o4Rrfmu5ZHhpt_y5UjbzPZSzJsHO2klK?usp=drive_link
    drive_target_id: str = "1roEruPwNWxBt8lJUGib4_M_ehl9Shj64" # https://drive.google.com/drive/folders/1roEruPwNWxBt8lJUGib4_M_ehl9Shj64?usp=drive_link
    run_drive_shadow: bool = True

    # 3. Local paths nhìn từ Colab sau khi mount Google Drive.
    # RẤT QUAN TRỌNG: archive_source_dir phải là local mount path tương ứng với drive_target_id.
    # drive-shadow copy vào drive_target_id; standardize đọc từ archive_source_dir.
    archive_source_dir: str = "/content/drive/MyDrive/AIC2026/raw_dataset"
    archive_target_dir: str = "/content/drive/MyDrive/AIC2026/standardize"

    # Temp phải nằm trên local runtime để giảm ghi/cache qua Google DriveFS.
    # Package phải cleanup temp sau từng member/source và throttle khi disk thấp.
    # Không đặt temp_extract_dir trong /content/drive cho dataset lớn.
    temp_extract_dir: str = "/content/temp_extract"

    # 4. Repo code
    github_repo_url: str = "https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git"
    github_branch: str = "system1-optimized"
    repo_dir_name: str = "Multimodal-Agentic-Retrieval-Engine"

    # 5. Execution options
    execution_mode: str = "bronze_fast"
    num_batches: int = 10
    run_standardize_archives: bool = True
    run_ingest: bool = True
    run_assign_batches: bool = True

    # Upload processed artifacts nhỏ sau khi chia batch.
    run_upload_processed: bool = True

    def __post_init__(self):
        self.env = "colab" if "google.colab" in sys.modules else "local"
        self.workspace = Path("/content") if self.env == "colab" else Path.cwd()
        os.environ["AIC_RELEASE_ID"] = self.release_id
        os.environ["AIC_HF_REPO_ID"] = self.hf_release_repo

config = WorkflowConfig()

print("Môi trường:", config.env)
print("Workspace:", config.workspace)
print("Release ID:", config.release_id)
print("HF processed repo:", config.hf_release_repo)
print("HF raw repo:", config.hf_canonical_repo)
print("Raw import ID:", config.raw_import_id)
print("Run upload standardized raw:", config.run_upload_standardized_raw)
print("Run drive shadow:", config.run_drive_shadow)
print("Drive source ID:", config.drive_source_id)
print("Drive target ID:", config.drive_target_id)
print("Archive source dir dùng để standardize:", config.archive_source_dir)
print("Archive target dir:", config.archive_target_dir)
print("Temp extract dir:", config.temp_extract_dir)


Môi trường: colab
Workspace: /content
Release ID: canonical_release_v002
HF processed repo: 1thesudden/AIC2026_processed
HF raw repo: 1thesudden/AIC2026_raw
Raw import ID: canonical_dataset_v002
Run upload standardized raw: True
Run drive shadow: True
Drive source ID: 1o4Rrfmu5ZHhpt_y5UjbzPZSzJsHO2klK
Drive target ID: 1roEruPwNWxBt8lJUGib4_M_ehl9Shj64
Archive source dir dùng để standardize: /content/drive/MyDrive/AIC2026/raw_dataset
Archive target dir: /content/drive/MyDrive/AIC2026/standardize
Temp extract dir: /content/temp_extract


In [ ]:
# BƯỚC 1: Mount Google Drive, authenticate Google, và lấy HF token.
import os
import sys
import shutil
import subprocess
from pathlib import Path

# 1. Mount Drive và lấy HF token.
if config.env == "colab":
    from google.colab import userdata, drive, auth

    hf_token = userdata.get("HF_TOKEN")
    if not hf_token:
        raise RuntimeError("Thiếu HF_TOKEN trong Colab Secrets. Hãy thêm HF_TOKEN trước khi chạy notebook.")

    os.environ["HF_TOKEN"] = hf_token
    os.environ["AIC_HF_TOKEN"] = hf_token

    # Giữ output Colab sạch hơn khi dùng huggingface_hub trong subprocess.
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
    os.environ["HF_HUB_VERBOSITY"] = "error"
    os.environ.setdefault("AIC_VERBOSE", "0")
    os.environ.setdefault("AIC_HF_PROGRESS", "0")

    drive.mount("/content/drive", force_remount=False)
    auth.authenticate_user()
else:
    if not os.environ.get("HF_TOKEN") and not os.environ.get("AIC_HF_TOKEN"):
        print("Cảnh báo: chưa thấy HF_TOKEN/AIC_HF_TOKEN trong environment.")

    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
    os.environ["HF_HUB_VERBOSITY"] = "error"
    os.environ.setdefault("AIC_VERBOSE", "0")
    os.environ.setdefault("AIC_HF_PROGRESS", "0")


In [31]:
# BƯỚC 2: Clone hoặc sync repo code.
repo_dir = Path(config.workspace) / config.repo_dir_name
repo_dir = repo_dir.expanduser().resolve()

# Quan trọng: nếu kernel đang đứng trong repo cũ đã bị xóa, phải cd về thư mục còn tồn tại.
os.chdir("/content")

print("Current cwd:", Path.cwd())
print("repo_dir:", repo_dir)
print("repo_dir exists:", repo_dir.exists())
print("repo_dir .git exists:", (repo_dir / ".git").exists())
print("github_repo_url:", config.github_repo_url)
print("github_branch:", config.github_branch)

def run_git(cmd, *, cwd=None, check=True):
    safe_cwd = Path(cwd).expanduser().resolve() if cwd else Path("/content")
    safe_cwd.mkdir(parents=True, exist_ok=True)

    print("CWD:", safe_cwd)
    print("RUN:", " ".join(map(str, cmd)))

    result = subprocess.run(
        list(map(str, cmd)),
        cwd=str(safe_cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if result.stdout:
        print(result.stdout)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit_code={result.returncode}\n"
            f"CMD: {' '.join(map(str, cmd))}\n\n"
            f"OUTPUT:\n{result.stdout}"
        )

    return result

if (repo_dir / ".git").exists():
    print("Repo đã tồn tại, cập nhật repo:", repo_dir)

    run_git(["git", "fetch", "--all", "--prune"], cwd=repo_dir)
    run_git(["git", "branch", "-a"], cwd=repo_dir, check=False)

    branch_check = run_git(
        ["git", "rev-parse", "--verify", f"origin/{config.github_branch}"],
        cwd=repo_dir,
        check=False,
    )

    if branch_check.returncode != 0:
        raise RuntimeError(
            f"Không tìm thấy remote branch origin/{config.github_branch}. "
            "Có thể agent chưa push commit/branch, hoặc config.github_branch sai."
        )

    run_git(
        ["git", "checkout", "-B", config.github_branch, f"origin/{config.github_branch}"],
        cwd=repo_dir,
    )

    run_git(
        ["git", "reset", "--hard", f"origin/{config.github_branch}"],
        cwd=repo_dir,
    )

else:
    if repo_dir.exists():
        print("Path tồn tại nhưng không phải git repo, xóa sạch:", repo_dir)
        shutil.rmtree(repo_dir, ignore_errors=True)

        if repo_dir.exists():
            print("shutil.rmtree chưa xóa sạch, dùng rm -rf:", repo_dir)
            subprocess.run(["rm", "-rf", str(repo_dir)], cwd="/content", check=True)

        if repo_dir.exists():
            raise RuntimeError(f"Không xóa được repo_dir: {repo_dir}")

    print("Clone repo:", config.github_repo_url)

    run_git([
        "git", "clone",
        config.github_repo_url,
        str(repo_dir),
    ], cwd="/content")

    run_git(["git", "fetch", "--all", "--prune"], cwd=repo_dir)
    run_git(["git", "branch", "-a"], cwd=repo_dir, check=False)

    branch_check = run_git(
        ["git", "rev-parse", "--verify", f"origin/{config.github_branch}"],
        cwd=repo_dir,
        check=False,
    )

    if branch_check.returncode != 0:
        raise RuntimeError(
            f"Clone được repo nhưng không tìm thấy remote branch origin/{config.github_branch}. "
            "Hãy kiểm tra agent đã push branch/commit chưa, hoặc đổi config.github_branch."
        )

    run_git(
        ["git", "checkout", "-B", config.github_branch, f"origin/{config.github_branch}"],
        cwd=repo_dir,
    )

print("Git commit hiện tại:")
run_git(["git", "log", "-1", "--oneline"], cwd=repo_dir)

print("Git status:")
run_git(["git", "status", "--short"], cwd=repo_dir)

Current cwd: /content
repo_dir: /content/Multimodal-Agentic-Retrieval-Engine
repo_dir exists: False
repo_dir .git exists: False
github_repo_url: https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git
github_branch: system1-optimized
Clone repo: https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git
CWD: /content
RUN: git clone https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git /content/Multimodal-Agentic-Retrieval-Engine
Cloning into '/content/Multimodal-Agentic-Retrieval-Engine'...

CWD: /content/Multimodal-Agentic-Retrieval-Engine
RUN: git fetch --all --prune
Fetching origin

CWD: /content/Multimodal-Agentic-Retrieval-Engine
RUN: git branch -a
* master
  remotes/origin/HEAD -> origin/master
  remotes/origin/dev
  remotes/origin/implementation/web-retrieval-system
  remotes/origin/master
  remotes/origin/system1
  remotes/origin/system1-optimized

CWD: /content/Multimodal-Agentic-Retrieval-Engine
RUN: git rev-parse --verify origin/system1-opt

CompletedProcess(args=['git', 'status', '--short'], returncode=0, stdout='')

In [32]:
# BƯỚC 2B OPTIONAL: Kiểm tra nhanh branch/commit hiện tại.
# Cell này chỉ để debug. Cell BƯỚC 2 đã in git log/status rồi.
%cd {repo_dir}
!git branch --show-current
!git log -1 --oneline
!git status --short


/content/Multimodal-Agentic-Retrieval-Engine
system1-optimized
1b0c636 (HEAD -> system1-optimized, origin/system1-optimized) Document Notebook 00 phase workflow


In [13]:
# BƯỚC 3: Định nghĩa helper run_cli để gọi system1 CLI trong notebook.
def run_cli(args, *, check=True, stream=True, tail_lines=300):
    import os
    import subprocess
    import sys
    from pathlib import Path

    def find_repo_root():
        candidates = [
            globals().get("SYSTEM1_ROOT"),
            globals().get("REPO_ROOT"),
            Path.cwd(),
            Path("/content/Multimodal-Agentic-Retrieval-Engine"),
        ]

        for candidate in candidates:
            if candidate is None:
                continue
            path = Path(candidate).expanduser().resolve()

            # Nếu đang ở project package system1/.
            if path.name == "system1" and (path / "src" / "system1").exists():
                return path

            # Nếu đang ở repo root có folder system1/src/system1.
            if (path / "system1" / "src" / "system1").exists():
                return path

        raise RuntimeError(
            "Không tìm thấy repo root. Hãy chạy cell clone/setup repo trước, "
            "hoặc kiểm tra repo có nằm ở /content/Multimodal-Agentic-Retrieval-Engine không."
        )

    cli_cwd = find_repo_root()
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    # Giữ tương thích với bản cũ:
    # - nếu cli_cwd là repo root cha: dùng system1/src
    # - nếu cli_cwd là package root system1/: dùng src
    if (cli_cwd / "src" / "system1").exists():
        system1_src = cli_cwd / "src"
    else:
        system1_src = cli_cwd / "system1" / "src"

    env["PYTHONPATH"] = str(system1_src) + os.pathsep + env.get("PYTHONPATH", "")

    # Chặn progress bar nội bộ của huggingface_hub trong subprocess.
    # Nếu cần debug HF progress bar thật, set AIC_HF_PROGRESS=1 trước khi gọi run_cli.
    if env.get("AIC_HF_PROGRESS", "0") != "1":
        env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
        env["HF_HUB_VERBOSITY"] = "error"

    # Mặc định không in per-file/per-cleanup nếu package đã hỗ trợ AIC_VERBOSE.
    env.setdefault("AIC_VERBOSE", "0")

    cmd = [sys.executable, "-m", "system1.cli", *args]

    print("\n" + "=" * 100)
    print("CWD:", cli_cwd)
    print("PYTHONPATH prefix:", system1_src)
    print("RUN:", " ".join(cmd))
    print("stream:", stream)
    print("=" * 100)

    if not stream:
        completed = subprocess.run(
            cmd,
            cwd=str(cli_cwd),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            check=False,
        )

        output = completed.stdout or ""
        output_lines = output.splitlines()
        tail = "\n".join(output_lines[-tail_lines:])

        print(f"CLI finished: exit_code={completed.returncode}")
        print(f"Last {min(tail_lines, len(output_lines))} lines:")
        print("-" * 100)
        print(tail)
        print("-" * 100)

        if check and completed.returncode != 0:
            raise RuntimeError(
                f"CLI failed with exit code {completed.returncode}: {' '.join(args)}\n\n"
                f"Last output:\n{tail}"
            )

        return output

    process = subprocess.Popen(
        cmd,
        cwd=str(cli_cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)

    returncode = process.wait()
    output = "".join(lines)

    if check and returncode != 0:
        tail = "\n".join(output.splitlines()[-tail_lines:])
        raise RuntimeError(
            f"CLI failed with exit code {returncode}: {' '.join(args)}\n\n"
            f"Last output:\n{tail}"
        )

    return output

In [5]:
# BƯỚC 4: Resolve runtime paths và kiểm tra mapping Drive workflow.
from system1.runtime.environment import resolve_runtime_paths
from pathlib import Path

runtime_paths = resolve_runtime_paths(output_root=config.workspace / "output")
output_base = runtime_paths.output_root
output_base.mkdir(parents=True, exist_ok=True)

print("Runtime paths:")
print("- environment:", runtime_paths.environment)
print("- workspace_root:", runtime_paths.workspace_root)
print("- input_root:", runtime_paths.input_root)
print("- output_root:", runtime_paths.output_root)
print("- artifact_root:", runtime_paths.artifact_root)
print("- release_id:", runtime_paths.release_id)

archive_source = Path(config.archive_source_dir)
archive_target = Path(config.archive_target_dir)
temp_extract = Path(config.temp_extract_dir)

print("Workflow check:")
print("- drive-shadow sẽ copy vào drive_target_id:", config.drive_target_id)
print("- standardize sẽ đọc local archive_source_dir:", archive_source)
print("- Hai giá trị này phải trỏ tới cùng một folder Google Drive.")

print("Trạng thái archive_source_dir trước drive-shadow:")
print("- exists:", archive_source.exists())
if archive_source.exists():
    visible_items = list(archive_source.rglob("*"))
    print("- visible item count before shadow:", len(visible_items))
    for p in visible_items[:30]:
        print(" ", p)
else:
    print("- Chưa thấy folder local. Notebook vẫn sẽ chạy drive-shadow trước, sau đó remount Drive và kiểm tra lại.")

Runtime paths:
- environment: colab
- workspace_root: /content
- input_root: /content/input
- output_root: /content/output
- artifact_root: /content/system1_artifacts
- release_id: canonical_release_v002
Workflow check:
- drive-shadow sẽ copy vào drive_target_id: 1roEruPwNWxBt8lJUGib4_M_ehl9Shj64
- standardize sẽ đọc local archive_source_dir: /content/drive/MyDrive/AIC2026/raw_dataset
- Hai giá trị này phải trỏ tới cùng một folder Google Drive.
Trạng thái archive_source_dir trước drive-shadow:
- exists: True
- visible item count before shadow: 15
  /content/drive/MyDrive/AIC2026/raw_dataset/media-info-aic25-b1.zip
  /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L21_a.zip
  /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L22_a.zip
  /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L23_a.zip
  /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L24_a.zip
  /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L25_a1.zip
  /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L26_a.zip
  /co

In [6]:
# BƯỚC 5: Drive-shadow copy từ folder BTC/source sang folder Drive của team/bạn.
from pathlib import Path
import json
import time
from google.colab import drive

drive_shadow_report_path = Path(config.workspace) / "drive_shadow_report.json"
archive_source = Path(config.archive_source_dir)

if config.run_drive_shadow:
    if not config.drive_source_id or not config.drive_target_id:
        raise RuntimeError("run_drive_shadow=True nhưng thiếu drive_source_id hoặc drive_target_id.")

    print("Bắt đầu drive-shadow:")
    print("- source_folder_id:", config.drive_source_id)
    print("- dest_folder_id:", config.drive_target_id)
    print("- report_path:", drive_shadow_report_path)

    run_cli([
        "drive-shadow",
        "--source-folder-id", config.drive_source_id,
        "--dest-folder-id", config.drive_target_id,
        "--report-path", str(drive_shadow_report_path),
    ])
else:
    raise RuntimeError(
        "Workflow hiện tại yêu cầu chạy drive-shadow trước. "
        "Hãy để config.run_drive_shadow = True."
    )

Bắt đầu drive-shadow:
- source_folder_id: 1o4Rrfmu5ZHhpt_y5UjbzPZSzJsHO2klK
- dest_folder_id: 1roEruPwNWxBt8lJUGib4_M_ehl9Shj64
- report_path: /content/drive_shadow_report.json

CWD: /content/Multimodal-Agentic-Retrieval-Engine
PYTHONPATH prefix: /content/Multimodal-Agentic-Retrieval-Engine/system1/src
RUN: /usr/bin/python3 -m system1.cli drive-shadow --source-folder-id 1o4Rrfmu5ZHhpt_y5UjbzPZSzJsHO2klK --dest-folder-id 1roEruPwNWxBt8lJUGib4_M_ehl9Shj64 --report-path /content/drive_shadow_report.json
[drive-shadow] source_folder_id=1o4Rrfmu5ZHhpt_y5UjbzPZSzJsHO2klK dest_folder_id=1roEruPwNWxBt8lJUGib4_M_ehl9Shj64 report_path=/content/drive_shadow_report.json
httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
[drive-shadow] source folder: video_batch_1 (1o4Rrfmu5ZHhpt_y5UjbzPZSzJsHO2klK)
[drive-shadow] 

In [7]:
# BƯỚC 6: Kiểm tra report của drive-shadow trước khi đọc dữ liệu từ Drive mount.
from pathlib import Path
import json

drive_shadow_report_path = Path(config.workspace) / "drive_shadow_report.json"

print("Kiểm tra drive-shadow report:")
print("- path:", drive_shadow_report_path)
print("- exists:", drive_shadow_report_path.exists())

drive_shadow_report = None
copied_files = 0
created_folders = 0
skipped_existing = 0
skipped_google_apps = 0
error_count = None

if drive_shadow_report_path.exists():
    drive_shadow_report = json.loads(drive_shadow_report_path.read_text(encoding="utf-8"))

    copied_files = int(drive_shadow_report.get("copied_files", 0))
    created_folders = int(drive_shadow_report.get("created_folders", 0))
    skipped_existing = int(drive_shadow_report.get("skipped_existing", 0))
    skipped_google_apps = int(drive_shadow_report.get("skipped_google_apps", 0))
    error_count = int(drive_shadow_report.get("error_count", 0))

    print("Drive shadow summary:")
    print("- status:", drive_shadow_report.get("status"))
    print("- source_folder_id:", drive_shadow_report.get("source_folder_id"))
    print("- dest_folder_id:", drive_shadow_report.get("dest_folder_id"))
    print("- source_folder_name:", drive_shadow_report.get("source_folder_name"))
    print("- dest_folder_name:", drive_shadow_report.get("dest_folder_name"))
    print("- copied_files:", copied_files)
    print("- created_folders:", created_folders)
    print("- skipped_existing:", skipped_existing)
    print("- skipped_google_apps:", skipped_google_apps)
    print("- error_count:", error_count)

    if error_count and error_count > 0:
        print("\nMột số item lỗi đầu tiên:")
        failed_items = [
            item for item in drive_shadow_report.get("items", [])
            if item.get("status") == "failed"
        ]
        for item in failed_items[:20]:
            print(item)

        raise RuntimeError(
            "drive-shadow có error_count > 0. "
            "Không chạy standardize khi copy Drive chưa sạch lỗi."
        )

    if copied_files + created_folders + skipped_existing + skipped_google_apps == 0:
        raise RuntimeError(
            "drive-shadow chạy xong nhưng report cho thấy không copy/tạo/skip bất kỳ item nào. "
            "Kiểm tra source folder hoặc quyền Drive."
        )
else:
    raise FileNotFoundError(
        f"Không thấy drive-shadow report: {drive_shadow_report_path}. "
        "CLI drive-shadow cần tạo report sau khi chạy."
    )

Kiểm tra drive-shadow report:
- path: /content/drive_shadow_report.json
- exists: True
Drive shadow summary:
- status: pass
- source_folder_id: 1o4Rrfmu5ZHhpt_y5UjbzPZSzJsHO2klK
- dest_folder_id: 1roEruPwNWxBt8lJUGib4_M_ehl9Shj64
- source_folder_name: video_batch_1
- dest_folder_name: raw_dataset
- copied_files: 0
- created_folders: 0
- skipped_existing: 15
- skipped_google_apps: 0
- error_count: 0


In [8]:
# BƯỚC 7: Remount Google Drive và xác nhận Colab đã nhìn thấy đủ file.
from pathlib import Path
import json
import time
from google.colab import drive

archive_source = Path(config.archive_source_dir)
drive_shadow_report_path = Path(config.workspace) / "drive_shadow_report.json"

print("Remount Google Drive để Colab nhìn thấy dữ liệu mới...")

try:
    drive.flush_and_unmount()
except Exception as exc:
    print("flush_and_unmount warning:", exc)

time.sleep(5)
drive.mount("/content/drive", force_remount=True)

print("Kiểm tra local archive_source_dir sau drive-shadow:")
print("- archive_source_dir:", archive_source)
print("- exists:", archive_source.exists())

if not archive_source.exists():
    raise RuntimeError(
        "Không thấy archive_source_dir sau drive-shadow. "
        "archive_source_dir phải là local path tương ứng với drive_target_id."
    )

if not drive_shadow_report_path.exists():
    raise RuntimeError(f"Không thấy drive-shadow report để đối chiếu: {drive_shadow_report_path}")

drive_shadow_report = json.loads(drive_shadow_report_path.read_text(encoding="utf-8"))

# Chỉ các file thường đã copied/skipped_existing mới cần xuất hiện trong Drive mount.
# skipped_google_apps không phải file local; created_folders không tính vào file count.
expected_relative_files = sorted({
    str(item.get("path", "")).strip("/")
    for item in drive_shadow_report.get("items", [])
    if item.get("kind") == "file" and item.get("status") in {"copied", "skipped_existing"} and item.get("path")
})

expected_file_count_from_report = int(drive_shadow_report.get("copied_files", 0)) + int(drive_shadow_report.get("skipped_existing", 0))
expected_file_count = len(expected_relative_files) or expected_file_count_from_report

print("Drive mount sync expectation:")
print("- expected_file_count:", expected_file_count)
print("- expected_relative_files sample:", expected_relative_files[:20])

visible_files = []
visible_relative_files = set()
missing_expected = set(expected_relative_files)

# LỖI CŨ: cell dừng ngay khi thấy visible_items > 0, ví dụ mới thấy 4/15 file.
# SỬA: chỉ pass khi thấy đủ file theo drive_shadow_report.
for attempt in range(1, 31):
    visible_files = sorted([p for p in archive_source.rglob("*") if p.is_file()])
    visible_relative_files = {
        p.relative_to(archive_source).as_posix()
        for p in visible_files
    }

    if expected_relative_files:
        missing_expected = set(expected_relative_files) - visible_relative_files
        ready = len(missing_expected) == 0
    else:
        missing_expected = set()
        ready = len(visible_files) >= expected_file_count if expected_file_count else len(visible_files) > 0

    print(
        f"- attempt {attempt}/30: "
        f"visible_file_count={len(visible_files)} "
        f"expected_file_count={expected_file_count} "
        f"missing_expected={len(missing_expected)}"
    )

    if ready:
        break

    if missing_expected:
        print("  missing sample:", sorted(missing_expected)[:10])

    time.sleep(10)

if expected_file_count and len(visible_files) < expected_file_count:
    raise RuntimeError(
        f"Drive mount mới thấy {len(visible_files)}/{expected_file_count} file sau drive-shadow. "
        "Không chạy standardize vì sẽ chỉ xử lý một phần dataset. "
        "Đây thường là lỗi sync/cache của Google Drive mount trong Colab. "
        "Hãy đợi thêm vài phút rồi chạy lại riêng cell kiểm tra này."
    )

if expected_relative_files and missing_expected:
    raise RuntimeError(
        f"Drive mount vẫn thiếu {len(missing_expected)} file theo drive-shadow report. "
        f"Ví dụ thiếu: {sorted(missing_expected)[:20]}. "
        "Không chạy standardize khi local mount chưa thấy đủ file."
    )

print("Local Drive mount đã thấy đủ dữ liệu theo drive-shadow report:")
print("- visible_file_count:", len(visible_files))
for p in visible_files[:50]:
    print(" ", p)


Remount Google Drive để Colab nhìn thấy dữ liệu mới...
Mounted at /content/drive
Kiểm tra local archive_source_dir sau drive-shadow:
- archive_source_dir: /content/drive/MyDrive/AIC2026/raw_dataset
- exists: True
Drive mount sync expectation:
- expected_file_count: 15
- expected_relative_files sample: ['Videos_L21_a.zip', 'Videos_L22_a.zip', 'Videos_L23_a.zip', 'Videos_L24_a.zip', 'Videos_L25_a1.zip', 'Videos_L26_a.zip', 'Videos_L26_b.zip', 'Videos_L26_c.zip', 'Videos_L26_d.zip', 'Videos_L26_e.zip', 'Videos_L27_a.zip', 'Videos_L28_a.zip', 'Videos_L29_a.zip', 'Videos_L30_a.zip', 'media-info-aic25-b1.zip']
- attempt 1/30: visible_file_count=15 expected_file_count=15 missing_expected=0
Local Drive mount đã thấy đủ dữ liệu theo drive-shadow report:
- visible_file_count: 15
  /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L21_a.zip
  /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L22_a.zip
  /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L23_a.zip
  /content/drive/MyDrive/AIC2026/r

In [9]:
# BƯỚC 8: Standardize dữ liệu thành raw_videos/metadata và tạo pairing audit reports.
# Dùng local temp để giảm DriveFS cache và dùng guard/throttle trong package.
# Cell này cũng ghi missing_metadata.json và unmatched_metadata.json từ pairing audit.

from pathlib import Path
import json
import shutil
import subprocess

archive_source = Path(config.archive_source_dir)
standardized_root = Path(config.archive_target_dir)
temp_extract = Path(config.temp_extract_dir)
drive_shadow_report_path = Path(config.workspace) / "drive_shadow_report.json"

raw_video_dir = standardized_root / "raw_videos"
metadata_dir = standardized_root / "metadata"
progress_path = standardized_root / "standardize_progress.jsonl"
report_path = standardized_root / "standardize_archives_report.json"

print("BƯỚC 8: Standardize with local temp + DriveFS-safe guards")
print("- archive_source:", archive_source)
print("- standardized_root:", standardized_root)
print("- temp_extract:", temp_extract)
print("- progress_path:", progress_path)
print("- report_path:", report_path)

if not archive_source.exists():
    raise RuntimeError(f"Không thấy archive_source_dir: {archive_source}")

# Với dataset lớn, temp phải nằm ở local runtime để giảm 1 lượt ghi qua Google DriveFS.
# Không dùng /content/drive làm temp vì DriveFS sẽ tạo cache/upload buffer nhiều lần.
temp_extract_str = str(temp_extract.resolve() if temp_extract.exists() else temp_extract)
if temp_extract_str.startswith("/content/drive/"):
    raise RuntimeError(
        f"temp_extract đang nằm trên Google Drive mount: {temp_extract}. "
        "Với dataset lớn, hãy đặt config.temp_extract_dir = '/content/aic_scratch'."
    )

if not temp_extract_str.startswith("/content/"):
    print(
        "WARNING: temp_extract không nằm dưới /content. "
        "Trên Colab nên dùng /content/aic_scratch để cleanup nhanh và tránh DriveFS temp."
    )

standardized_root.mkdir(parents=True, exist_ok=True)

# Cleanup scratch trước khi chạy để không kế thừa rác từ lần run trước.
# Chỉ xóa temp_extract local, không xóa source/output trên Drive.
if temp_extract.exists():
    print("Cleanup local temp_extract trước khi standardize:", temp_extract)
    shutil.rmtree(temp_extract, ignore_errors=True)
temp_extract.mkdir(parents=True, exist_ok=True)

print("Disk trước standardize:")
subprocess.run("df -h / /content /content/drive 2>/dev/null || df -h / /content", shell=True, check=False)
subprocess.run(f"du -sh {str(temp_extract)!r} 2>/dev/null || true", shell=True, check=False)

source_files = sorted([p for p in archive_source.rglob("*") if p.is_file()])
print("- source file count:", len(source_files))
for p in source_files[:30]:
    print(" source:", p)

if len(source_files) == 0:
    raise RuntimeError("archive_source_dir đang rỗng. Không chạy standardize.")

# Guard chống lỗi Drive mount chỉ thấy một phần file sau drive-shadow.
# Nếu report nói copy/skip 15 file mà local mount mới thấy 4, dừng ngay tại đây.
if drive_shadow_report_path.exists():
    drive_shadow_report = json.loads(drive_shadow_report_path.read_text(encoding="utf-8"))
    expected_relative_files = sorted({
        str(item.get("path", "")).strip("/")
        for item in drive_shadow_report.get("items", [])
        if item.get("kind") == "file" and item.get("status") in {"copied", "skipped_existing"} and item.get("path")
    })
    expected_file_count = len(expected_relative_files) or (
        int(drive_shadow_report.get("copied_files", 0)) + int(drive_shadow_report.get("skipped_existing", 0))
    )

    if expected_relative_files:
        source_relative_files = {p.relative_to(archive_source).as_posix() for p in source_files}
        missing_expected = sorted(set(expected_relative_files) - source_relative_files)
        if missing_expected:
            raise RuntimeError(
                f"archive_source_dir chỉ thấy {len(source_files)} file nhưng còn thiếu "
                f"{len(missing_expected)} file theo drive-shadow report. "
                f"Ví dụ thiếu: {missing_expected[:20]}. "
                "Không chạy standardize vì sẽ tạo dataset thiếu. "
                "Hãy chạy lại cell remount/kiểm tra Drive cho tới khi đủ file."
            )
    elif expected_file_count and len(source_files) < expected_file_count:
        raise RuntimeError(
            f"archive_source_dir chỉ thấy {len(source_files)}/{expected_file_count} file theo drive-shadow report. "
            "Không chạy standardize vì sẽ tạo dataset thiếu. "
            "Hãy chạy lại cell remount/kiểm tra Drive cho tới khi đủ file."
        )

# Fail sớm nếu package chưa có các option disk-safe mới.
help_result = subprocess.run(
    [sys.executable, "-m", "system1.cli", "standardize-archives", "--help"],
    text=True,
    capture_output=True,
    check=False,
)
help_text = (help_result.stdout or "") + "\n" + (help_result.stderr or "")
required_options = [
    "--min-free-gb",
    "--drive-sync-sleep-seconds",
    "--cleanup-every-files",
    "--cleanup-every-gb",
]
missing_options = [opt for opt in required_options if opt not in help_text]
if missing_options:
    raise RuntimeError(
        "Package hiện tại chưa hỗ trợ các option disk-safe cho standardize-archives: "
        f"{missing_options}. Hãy để agent sửa package trước khi chạy BƯỚC 2."
    )

if config.run_standardize_archives:
    run_cli([
        "standardize-archives",
        "--source-dir", str(archive_source),
        "--target-dir", str(standardized_root),
        "--temp-dir", str(temp_extract),
        "--resume",
        "--progress-path", str(progress_path),
        "--min-free-gb", "15",
        "--drive-sync-sleep-seconds", "30",
        "--cleanup-every-files", "1",
        "--cleanup-every-gb", "50",
    ], stream=False, tail_lines=500)
else:
    print("config.run_standardize_archives=False, skip BƯỚC 2.")

print("Disk sau standardize:")
subprocess.run("df -h / /content /content/drive 2>/dev/null || df -h / /content", shell=True, check=False)
subprocess.run(f"du -sh {str(temp_extract)!r} 2>/dev/null || true", shell=True, check=False)

print("Kiểm tra standardized_root:", standardized_root)
if not standardized_root.exists():
    raise FileNotFoundError(f"Không tồn tại archive_target_dir sau standardize: {standardized_root}")

print("- raw_videos exists:", raw_video_dir.exists())
print("- metadata exists:", metadata_dir.exists())
print("- progress exists:", progress_path.exists())
print("- report exists:", report_path.exists())

if not raw_video_dir.exists():
    raise RuntimeError(f"Không tìm thấy raw_videos sau standardize: {raw_video_dir}")
if not metadata_dir.exists():
    raise RuntimeError(f"Không tìm thấy metadata sau standardize: {metadata_dir}")

media_exts = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".wav"}
media_files = sorted([p for p in raw_video_dir.rglob("*") if p.is_file() and p.suffix.lower() in media_exts])
json_files = sorted([p for p in metadata_dir.rglob("*.json") if p.is_file()])

print("media_count:", len(media_files))
print("metadata_count:", len(json_files))

for p in media_files[:20]:
    print(" media:", p)
for p in json_files[:20]:
    print(" metadata:", p)

if len(media_files) == 0:
    raise RuntimeError("raw_videos tồn tại nhưng không có media file. Cần sửa standardize_archive_source hoặc archive_source_dir sai.")

# Không fail nếu metadata thừa hoặc thiếu metadata ở đây.
# Đây là audit của bước standardize/pairing:
# - video là nguồn chính cho canonical raw upload
# - video thiếu metadata sẽ được ghi vào missing_metadata.json
# - metadata không có video tương ứng sẽ được ghi vào unmatched_metadata.json
video_stems = {p.stem for p in media_files}
metadata_stems = {p.stem for p in json_files}
missing_meta = sorted(video_stems - metadata_stems)
unmatched_meta = sorted(metadata_stems - video_stems)

print("pairing quick check:")
print("- videos missing metadata:", len(missing_meta))
print("- metadata without video:", len(unmatched_meta))

if missing_meta:
    print("Ví dụ video thiếu metadata:", missing_meta[:30])
if unmatched_meta:
    print("Ví dụ metadata không có video tương ứng:", unmatched_meta[:30])

# Ghi audit reports từ bước standardize/pairing.
# Hai file này thuộc logic đối chiếu raw_videos ↔ metadata, không phải kết quả tự nhiên của HF ingest.
# Processed release sẽ copy hai file này vào manifests/ ở bước sau.
missing_metadata_report_path = standardized_root / "missing_metadata.json"
unmatched_metadata_report_path = standardized_root / "unmatched_metadata.json"

missing_metadata_report = {
    "kind": "missing_metadata",
    "source": "standardize_pairing_audit",
    "description": "Videos có raw video nhưng không tìm thấy metadata JSON cùng stem.",
    "count": len(missing_meta),
    "missing_metadata": missing_meta,
    "missing_video_ids": missing_meta,
    "raw_video_dir": str(raw_video_dir),
    "metadata_dir": str(metadata_dir),
}

unmatched_metadata_report = {
    "kind": "unmatched_metadata",
    "source": "standardize_pairing_audit",
    "description": "Metadata JSON tồn tại nhưng không có raw video cùng stem.",
    "count": len(unmatched_meta),
    "unmatched_metadata": unmatched_meta,
    "unmatched_metadata_ids": unmatched_meta,
    "raw_video_dir": str(raw_video_dir),
    "metadata_dir": str(metadata_dir),
}

missing_metadata_report_path.write_text(
    json.dumps(missing_metadata_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
unmatched_metadata_report_path.write_text(
    json.dumps(unmatched_metadata_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Pairing audit reports written:")
print("- missing_metadata:", missing_metadata_report_path)
print("- unmatched_metadata:", unmatched_metadata_report_path)

print("Standardize OK enough for tolerant ingest:")
print("- raw_video_dir:", raw_video_dir)
print("- metadata_dir:", metadata_dir)

BƯỚC 2: Standardize with local temp + DriveFS-safe guards
- archive_source: /content/drive/MyDrive/AIC2026/raw_dataset
- standardized_root: /content/drive/MyDrive/AIC2026/standardize
- temp_extract: /content/temp_extract
- progress_path: /content/drive/MyDrive/AIC2026/standardize/standardize_progress.jsonl
- report_path: /content/drive/MyDrive/AIC2026/standardize/standardize_archives_report.json
Disk trước standardize:
- source file count: 15
 source: /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L21_a.zip
 source: /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L22_a.zip
 source: /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L23_a.zip
 source: /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L24_a.zip
 source: /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L25_a1.zip
 source: /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L26_a.zip
 source: /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L26_b.zip
 source: /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L26_c.zip


In [ ]:
# BƯỚC 9: Upload standardized raw lên HF raw repo có version prefix.
# Upload raw_videos/metadata chuẩn hóa lên HF raw repo.
# Sau bước này, ingest sẽ đọc HF raw repo thay vì ingest local từ Drive.

from pathlib import Path

standardized_root = Path(config.archive_target_dir)

print("BƯỚC 9: Upload standardized raw")
print("- run_upload_standardized_raw:", config.run_upload_standardized_raw)
print("- standardized_root:", standardized_root)
print("- hf_canonical_repo:", config.hf_canonical_repo)
print("- raw_import_id:", config.raw_import_id)

if config.run_upload_standardized_raw:
    if not standardized_root.exists():
        raise RuntimeError(f"Không thấy standardized_root: {standardized_root}")
    if not (standardized_root / "raw_videos").exists():
        raise RuntimeError(f"Không thấy raw_videos: {standardized_root / 'raw_videos'}")
    if not (standardized_root / "metadata").exists():
        raise RuntimeError(f"Không thấy metadata: {standardized_root / 'metadata'}")

    run_cli([
        "upload-standardized-raw",
        "--source-dir", str(standardized_root),
        "--target-hf-repo-id", config.hf_canonical_repo,
        "--raw-import-id", config.raw_import_id,
    ], stream=False, tail_lines=300)
else:
    print("config.run_upload_standardized_raw=False, skip upload standardized raw.")

In [15]:
# BƯỚC 10: Kiểm tra HF raw repo sau upload-standardized-raw.
# Gate trước khi chạy HF ingest: raw repo phải đủ video/metadata + canonical manifests.
from huggingface_hub import HfApi, hf_hub_download
from pathlib import Path
import os, json

repo_id = config.hf_canonical_repo
prefix = config.raw_import_id.rstrip("/") + "/"
token = os.environ.get("HF_TOKEN") or os.environ.get("AIC_HF_TOKEN")
if not token:
    raise RuntimeError("Thiếu HF_TOKEN/AIC_HF_TOKEN để kiểm tra HF raw repo.")

api = HfApi(token=token)
files = set(api.list_repo_files(repo_id=repo_id, repo_type="dataset", token=token))

raw_videos = sorted(f for f in files if f.startswith(prefix + "raw_videos/"))
metadata = sorted(f for f in files if f.startswith(prefix + "metadata/"))

print("repo:", repo_id)
print("prefix:", prefix)
print("raw_videos:", len(raw_videos))
print("metadata:", len(metadata))
print("canonical_file_manifest:", prefix + "manifests/canonical_file_manifest.jsonl" in files)
print("canonical_import_report:", prefix + "manifests/canonical_import_report.json" in files)

if len(raw_videos) != 834 or len(metadata) != 834:
    raise RuntimeError("HF raw repo chưa đủ 834 video + 834 metadata. Chưa chạy upload-standardized-raw hoặc upload chưa hoàn tất.")

if prefix + "manifests/canonical_import_report.json" not in files:
    raise RuntimeError("Thiếu canonical_import_report.json.")

report_path = hf_hub_download(
    repo_id=repo_id,
    repo_type="dataset",
    filename=prefix + "manifests/canonical_import_report.json",
    token=token,
)

report = json.loads(Path(report_path).read_text(encoding="utf-8"))

print("report status:", report.get("status"))
print("video_count:", report.get("video_count"))
print("metadata_count:", report.get("metadata_count"))
print("uploaded_pair_count:", report.get("uploaded_pair_count"))
print("error_count:", report.get("error_count"))

if report.get("error_count", 0) != 0:
    print("WARNING: report còn lỗi. Nếu file count trên HF đã đủ, có thể rerun upload-standardized-raw để refresh report.")
else:
    print("HF raw OK. Có thể chạy ingest từ HF.")



repo: 1thesudden/AIC2026_raw
prefix: canonical_dataset_v002/
raw_videos: 834
metadata: 834
canonical_file_manifest: True
canonical_import_report: True
report status: pass
video_count: 834
metadata_count: 834
uploaded_pair_count: 834
error_count: 0
HF raw OK. Có thể chạy ingest từ HF.


In [33]:
# BƯỚC 11: Ingest từ HF raw repo để tạo processed artifacts local.
# Lưu ý: đây là HF canonical ingest, không phải local ingest từ Drive.
from pathlib import Path
import os
import shutil

release_root = output_base / config.release_id
canonical_staging_root = Path("/content/canonical_staging")

print("BƯỚC 11: Ingest từ HF raw repo")
print("- hf_canonical_repo:", config.hf_canonical_repo)
print("- raw_import_id:", config.raw_import_id)
print("- canonical_staging_root:", canonical_staging_root)
print("- release_root:", release_root)

if canonical_staging_root.exists():
    shutil.rmtree(canonical_staging_root, ignore_errors=True)
canonical_staging_root.mkdir(parents=True, exist_ok=True)

INGEST_MAX_WORKERS = "1"
os.environ["AIC_INGEST_MAX_WORKERS"] = INGEST_MAX_WORKERS
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"

run_cli([
    "ingest",
    "--mode", config.execution_mode,
    "--output", str(output_base),
    "--canonical-hf-repo-id", config.hf_canonical_repo,
    "--canonical-hf-prefix", config.raw_import_id,
    "--canonical-staging-root", str(canonical_staging_root),
    "--max-workers", INGEST_MAX_WORKERS,
    "--no-resume",
], stream=False, tail_lines=300)

videos_found = sorted(output_base.rglob("videos.parquet"))
media_manifest_found = sorted(output_base.rglob("media_store_manifest.parquet"))

print("videos.parquet found:", videos_found)
print("media_store_manifest.parquet found:", media_manifest_found)

if not videos_found:
    raise RuntimeError("Không tìm thấy videos.parquet sau ingest.")
if not media_manifest_found:
    raise RuntimeError("Không tìm thấy media_store_manifest.parquet sau ingest.")

BƯỚC 3: Ingest từ HF raw repo
- hf_canonical_repo: 1thesudden/AIC2026_raw
- raw_import_id: canonical_dataset_v002
- canonical_staging_root: /content/canonical_staging
- release_root: /content/output/canonical_release_v002

CWD: /content/Multimodal-Agentic-Retrieval-Engine
PYTHONPATH prefix: /content/Multimodal-Agentic-Retrieval-Engine/system1/src
RUN: /usr/bin/python3 -m system1.cli ingest --mode bronze_fast --output /content/output --canonical-hf-repo-id 1thesudden/AIC2026_raw --canonical-hf-prefix canonical_dataset_v002 --canonical-staging-root /content/canonical_staging --max-workers 1 --no-resume
stream: False
CLI finished: exit_code=0
Last 2 lines:
----------------------------------------------------------------------------------------------------
[ingest] source_backend=hf_dataset repo_id=1thesudden/AIC2026_raw prefix=canonical_dataset_v002 max_workers=1
Ingested sample inputs: /content/output/canonical_release_v002/manifests/dataset_report.json
----------------------------------

In [34]:
# BƯỚC 12: Chia batch từ videos.parquet đã tạo sau ingest.
# Lưu ý: workflow hiện tại tạo videos.parquet từ HF raw ingest.
# Command assign-batches hiện tại tự đọc từ output_base theo release/mode.

print("BƯỚC 12: Assign batches")
print("- output_base:", output_base)
print("- execution_mode:", config.execution_mode)
print("- num_batches:", config.num_batches)

videos_found = sorted(output_base.rglob("videos.parquet"))
media_manifest_found = sorted(output_base.rglob("media_store_manifest.parquet"))

print("videos.parquet found:", videos_found)
print("media_store_manifest.parquet found:", media_manifest_found)

if not videos_found:
    raise RuntimeError("Chưa có videos.parquet. Hãy chạy ingest thành công trước khi assign batches.")

if config.run_assign_batches:
    run_cli([
        "assign-batches",
        "--mode", config.execution_mode,
        "--num-batches", str(config.num_batches),
        "--output", str(output_base),
        "--no-resume",
    ], stream=False, tail_lines=300)
else:
    print("Bỏ qua assign-batches theo config.")

batch_manifest_found = sorted(output_base.rglob("batch_manifest.csv"))
batch_txt_found = sorted(output_base.rglob("batch_*.txt"))

print("batch_manifest.csv found:", batch_manifest_found)
print("batch_*.txt found:", batch_txt_found[:30])

if not batch_manifest_found:
    raise RuntimeError("Không tìm thấy batch_manifest.csv sau assign-batches.")

if not batch_txt_found:
    raise RuntimeError("Không tìm thấy batch_*.txt sau assign-batches.")

if len(batch_txt_found) != config.num_batches:
    raise RuntimeError(
        f"Số batch_*.txt không khớp: expected={config.num_batches}, actual={len(batch_txt_found)}"
    )

print("Assign batches OK.")

BƯỚC 4: Assign batches
- output_base: /content/output
- execution_mode: bronze_fast
- num_batches: 10
videos.parquet found: [PosixPath('/content/output/canonical_release_v002/tables/videos.parquet')]
media_store_manifest.parquet found: [PosixPath('/content/output/canonical_release_v002/raw_mapping/media_store_manifest.parquet')]

CWD: /content/Multimodal-Agentic-Retrieval-Engine
PYTHONPATH prefix: /content/Multimodal-Agentic-Retrieval-Engine/system1/src
RUN: /usr/bin/python3 -m system1.cli assign-batches --mode bronze_fast --num-batches 10 --output /content/output --no-resume
stream: False
CLI finished: exit_code=0
Last 2 lines:
----------------------------------------------------------------------------------------------------
Assigned batches: /content/output/canonical_release_v002/manifests
Saved checkpoint: /content/system1_artifacts/checkpoints/phase00_ingest_assignment.zip
----------------------------------------------------------------------------------------------------
batch_m

In [38]:
# BƯỚC 13: Gom audit reports vào processed release manifests.
# missing_metadata.json và unmatched_metadata.json được tạo ở BƯỚC 8 từ pairing audit.

from pathlib import Path
import shutil

release_root = output_base / config.release_id
manifests_root = release_root / "manifests"
manifests_root.mkdir(parents=True, exist_ok=True)

extra_reports = [
    Path(config.workspace) / "drive_shadow_report.json",
    Path(config.archive_target_dir) / "standardize_archives_report.json",
    Path(config.archive_target_dir) / "standardize_progress.jsonl",
    Path(config.archive_target_dir) / "missing_metadata.json",
    Path(config.archive_target_dir) / "unmatched_metadata.json",
]

for report_path in extra_reports:
    if report_path.exists():
        target = manifests_root / report_path.name
        shutil.copy2(report_path, target)
        print("Copied report:", report_path, "->", target)
    else:
        print("Report not found, skip:", report_path)


print("manifests:")
for p in sorted(manifests_root.iterdir()):
    print(p)
print("Release root:", release_root)
print("Release files:")
for p in sorted(release_root.rglob("*"))[:100]:
    print(" ", p)

Copied report: /content/drive_shadow_report.json -> /content/output/canonical_release_v002/manifests/drive_shadow_report.json
Copied report: /content/drive/MyDrive/AIC2026/standardize/standardize_archives_report.json -> /content/output/canonical_release_v002/manifests/standardize_archives_report.json
Copied report: /content/drive/MyDrive/AIC2026/standardize/standardize_progress.jsonl -> /content/output/canonical_release_v002/manifests/standardize_progress.jsonl
manifests:
/content/output/canonical_release_v002/manifests/batch_000.txt
/content/output/canonical_release_v002/manifests/batch_001.txt
/content/output/canonical_release_v002/manifests/batch_002.txt
/content/output/canonical_release_v002/manifests/batch_003.txt
/content/output/canonical_release_v002/manifests/batch_004.txt
/content/output/canonical_release_v002/manifests/batch_005.txt
/content/output/canonical_release_v002/manifests/batch_006.txt
/content/output/canonical_release_v002/manifests/batch_007.txt
/content/output/can

In [39]:
# BƯỚC 14: Upload processed artifacts nhỏ lên Hugging Face processed repo.
# Không upload raw_videos ở bước này.
# Lưu ý: raw_videos/metadata thật đã nằm trong HF raw repo.
# Repo processed chỉ chứa bảng, manifest, batch files và report nhỏ.

from pathlib import Path
import os

# Giữ output Colab sạch hơn khi dùng huggingface_hub.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"

try:
    from huggingface_hub import HfApi
except ImportError:
    %pip install -q huggingface_hub
    from huggingface_hub import HfApi

release_root = output_base / config.release_id
processed_repo_id = config.hf_release_repo

print("BƯỚC 14: Upload processed artifacts")
print("- run_upload_processed:", config.run_upload_processed)
print("- processed_repo_id:", processed_repo_id)
print("- release_root:", release_root)

if not config.run_upload_processed:
    print("Bỏ qua upload processed artifacts theo config.")
else:
    if not release_root.exists():
        raise RuntimeError(f"Không thấy release_root: {release_root}")

    hf_token = (
        os.environ.get("AIC_HF_TOKEN")
        or os.environ.get("HF_TOKEN")
    )

    if not hf_token:
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
            if hf_token:
                os.environ["HF_TOKEN"] = hf_token
                os.environ["AIC_HF_TOKEN"] = hf_token
        except Exception:
            pass

    if not hf_token:
        raise RuntimeError("Thiếu HF_TOKEN/AIC_HF_TOKEN để upload processed artifacts.")

    api = HfApi(token=hf_token)

    upload_patterns = [
        "tables/videos.parquet",
        "raw_mapping/media_store_manifest.parquet",
        "manifests/dataset_report.json",
        "manifests/ingestion_errors.jsonl",
        "manifests/missing_metadata.json",
        "manifests/unmatched_metadata.json",
        "manifests/batch_manifest.csv",
        "manifests/batch_*.txt",
        "manifests/drive_shadow_report.json",
        "manifests/standardize_archives_report.json",
        "manifests/standardize_progress.jsonl",
    ]

    files_to_upload = []

    for pattern in upload_patterns:
        files_to_upload.extend(sorted(release_root.glob(pattern)))

    # De-duplicate
    files_to_upload = sorted(set(files_to_upload))

    print("- file_count:", len(files_to_upload))

    if not files_to_upload:
        raise RuntimeError("Không có processed artifact nào để upload.")

    print("Files sẽ upload:")
    for local_path in files_to_upload:
        relative_path = local_path.relative_to(release_root).as_posix()
        remote_path = f"{config.release_id}/{relative_path}"
        size_kb = local_path.stat().st_size / 1024
        print(f"- {relative_path} -> {remote_path} ({size_kb:.1f} KB)")

    for local_path in files_to_upload:
        relative_path = local_path.relative_to(release_root).as_posix()
        remote_path = f"{config.release_id}/{relative_path}"

        api.upload_file(
            path_or_fileobj=str(local_path),
            path_in_repo=remote_path,
            repo_id=processed_repo_id,
            repo_type="dataset",
            token=hf_token,
        )

    print("Uploaded processed artifacts to HF:", processed_repo_id)

BƯỚC 6: Upload processed artifacts
- run_upload_processed: True
- processed_repo_id: 1thesudden/AIC2026_processed
- release_root: /content/output/canonical_release_v002
- file_count: 18
Files sẽ upload:
- manifests/batch_000.txt -> canonical_release_v002/manifests/batch_000.txt (0.7 KB)
- manifests/batch_001.txt -> canonical_release_v002/manifests/batch_001.txt (0.7 KB)
- manifests/batch_002.txt -> canonical_release_v002/manifests/batch_002.txt (0.7 KB)
- manifests/batch_003.txt -> canonical_release_v002/manifests/batch_003.txt (0.7 KB)
- manifests/batch_004.txt -> canonical_release_v002/manifests/batch_004.txt (0.7 KB)
- manifests/batch_005.txt -> canonical_release_v002/manifests/batch_005.txt (0.7 KB)
- manifests/batch_006.txt -> canonical_release_v002/manifests/batch_006.txt (0.7 KB)
- manifests/batch_007.txt -> canonical_release_v002/manifests/batch_007.txt (0.7 KB)
- manifests/batch_008.txt -> canonical_release_v002/manifests/batch_008.txt (0.7 KB)
- manifests/batch_009.txt -> can

In [40]:
# BƯỚC 15: Kiểm tra cấu trúc HF processed repo.
# Hai file missing/unmatched là required vì notebook BƯỚC 8 đã tạo và BƯỚC 13 đã copy vào release manifests.

from pathlib import Path
import os
import json
import pandas as pd

try:
    from huggingface_hub import HfApi, hf_hub_download
except ImportError:
    %pip install -q huggingface_hub
    from huggingface_hub import HfApi, hf_hub_download

repo_id = config.hf_release_repo
release_id = config.release_id

hf_token = (
    os.environ.get("AIC_HF_TOKEN")
    or os.environ.get("HF_TOKEN")
)

if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
        if hf_token:
            os.environ["HF_TOKEN"] = hf_token
            os.environ["AIC_HF_TOKEN"] = hf_token
    except Exception:
        pass

if not hf_token:
    raise RuntimeError("Thiếu HF_TOKEN/AIC_HF_TOKEN để kiểm tra HF repo.")

api = HfApi(token=hf_token)

files = api.list_repo_files(
    repo_id=repo_id,
    repo_type="dataset",
    token=hf_token,
)

release_files = sorted([
    f for f in files
    if f.startswith(f"{release_id}/")
])

print("HF processed repo:", repo_id)
print("release_id:", release_id)
print("release_file_count:", len(release_files))

for f in release_files:
    print(" ", f)

required_exact = [
    f"{release_id}/tables/videos.parquet",
    f"{release_id}/raw_mapping/media_store_manifest.parquet",
    f"{release_id}/manifests/dataset_report.json",
    f"{release_id}/manifests/ingestion_errors.jsonl",
    f"{release_id}/manifests/missing_metadata.json",
    f"{release_id}/manifests/unmatched_metadata.json",
    f"{release_id}/manifests/batch_manifest.csv",
    f"{release_id}/manifests/drive_shadow_report.json",
    f"{release_id}/manifests/standardize_archives_report.json",
    f"{release_id}/manifests/standardize_progress.jsonl",
]

missing_required = [p for p in required_exact if p not in release_files]

batch_txt_files = sorted([
    p for p in release_files
    if p.startswith(f"{release_id}/manifests/batch_") and p.endswith(".txt")
])

forbidden_patterns = [
    "/raw_videos/",
    "/metadata/",
    "/temp_extract/",
    "member_stage_",
    "member_extract_",
]

forbidden_files = [
    p for p in release_files
    if any(pattern in p for pattern in forbidden_patterns)
    or p.lower().endswith((".mp4", ".mov", ".mkv", ".avi", ".webm", ".wav", ".zip"))
]

print("\nCHECK REQUIRED:")
print("- missing_required:", missing_required)
print("- batch_txt_count:", len(batch_txt_files))
print("- batch_txt_files:", batch_txt_files)

print("\nCHECK FORBIDDEN:")
print("- forbidden_files:", forbidden_files)

if missing_required:
    raise RuntimeError(f"Thiếu required files trên HF processed repo: {missing_required}")

if len(batch_txt_files) == 0:
    raise RuntimeError("Không thấy manifests/batch_*.txt trên HF processed repo.")

if forbidden_files:
    raise RuntimeError(f"HF processed repo có file không nên upload: {forbidden_files}")

# Kiểm tra nội dung nếu raw upload đã bật: media_store_manifest phải có canonical columns.
if config.run_upload_standardized_raw:
    media_manifest_local = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=f"{release_id}/raw_mapping/media_store_manifest.parquet",
        token=hf_token,
    )
    media_df = pd.read_parquet(media_manifest_local)
    required_canonical_cols = {
        "canonical_backend",
        "canonical_repo_id",
        "canonical_repo_type",
        "canonical_import_id",
        "canonical_video_path",
        "canonical_metadata_path",
    }
    missing_canonical_cols = sorted(required_canonical_cols - set(media_df.columns))
    print("\nCHECK CANONICAL COLUMNS:")
    print("- missing_canonical_cols:", missing_canonical_cols)
    if missing_canonical_cols:
        raise RuntimeError(f"media_store_manifest thiếu canonical columns: {missing_canonical_cols}")

print("\nHF processed folder structure OK.")

HF processed repo: 1thesudden/AIC2026_processed
release_id: canonical_release_v002
release_file_count: 18
  canonical_release_v002/manifests/batch_000.txt
  canonical_release_v002/manifests/batch_001.txt
  canonical_release_v002/manifests/batch_002.txt
  canonical_release_v002/manifests/batch_003.txt
  canonical_release_v002/manifests/batch_004.txt
  canonical_release_v002/manifests/batch_005.txt
  canonical_release_v002/manifests/batch_006.txt
  canonical_release_v002/manifests/batch_007.txt
  canonical_release_v002/manifests/batch_008.txt
  canonical_release_v002/manifests/batch_009.txt
  canonical_release_v002/manifests/batch_manifest.csv
  canonical_release_v002/manifests/dataset_report.json
  canonical_release_v002/manifests/drive_shadow_report.json
  canonical_release_v002/manifests/ingestion_errors.jsonl
  canonical_release_v002/manifests/standardize_archives_report.json
  canonical_release_v002/manifests/standardize_progress.jsonl
  canonical_release_v002/raw_mapping/media_stor

RuntimeError: Thiếu required files trên HF processed repo: ['canonical_release_v002/manifests/missing_metadata.json', 'canonical_release_v002/manifests/unmatched_metadata.json']

In [41]:
# BƯỚC 16: Preview output của notebook 00.

from pathlib import Path
import pandas as pd

videos_found = sorted(output_base.rglob("videos.parquet"))
batch_manifest_found = sorted(output_base.rglob("batch_manifest.csv"))
media_manifest_found = sorted(output_base.rglob("media_store_manifest.parquet"))
batch_txt_found = sorted(output_base.rglob("batch_*.txt"))

print("Preview notebook 00 outputs")
print("- videos.parquet found:", videos_found)
print("- media_store_manifest.parquet found:", media_manifest_found)
print("- batch_manifest.csv found:", batch_manifest_found)
print("- batch_*.txt count:", len(batch_txt_found))

if not videos_found:
    raise RuntimeError("Không tìm thấy videos.parquet.")
if not media_manifest_found:
    raise RuntimeError("Không tìm thấy media_store_manifest.parquet.")
if not batch_manifest_found:
    raise RuntimeError("Không tìm thấy batch_manifest.csv.")
if not batch_txt_found:
    raise RuntimeError("Không tìm thấy batch_*.txt.")

videos_path = videos_found[0]
media_manifest_path = media_manifest_found[0]
batch_manifest_path = batch_manifest_found[0]

videos_df = pd.read_parquet(videos_path)
media_df = pd.read_parquet(media_manifest_path)
batch_df = pd.read_csv(batch_manifest_path)

print("\nvideos.parquet:", videos_path)
print("video_count:", len(videos_df))
display(videos_df.head())

print("\nmedia_store_manifest.parquet:", media_manifest_path)
print("media_manifest_rows:", len(media_df))
display(media_df.head())

print("\nbatch_manifest.csv:", batch_manifest_path)
print("batch_rows:", len(batch_df))
display(batch_df.head())

print("\nBatch txt files:")
for p in batch_txt_found[:30]:
    print(" ", p)

print("\nNotebook 00 hoàn tất nếu cell này chạy xong.")

Preview notebook 00 outputs
- videos.parquet found: [PosixPath('/content/output/canonical_release_v002/tables/videos.parquet')]
- media_store_manifest.parquet found: [PosixPath('/content/output/canonical_release_v002/raw_mapping/media_store_manifest.parquet')]
- batch_manifest.csv found: [PosixPath('/content/output/canonical_release_v002/manifests/batch_manifest.csv')]
- batch_*.txt count: 10

videos.parquet: /content/output/canonical_release_v002/tables/videos.parquet
video_count: 834


,video_id,video_ref,metadata_ref,source_filename,source_extension,fps_detected,fps_source,duration_seconds,width,height,frame_count,frame_count_estimated,frame_count_method,estimated_compute_cost,metadata_title
0,L21_V001,media://raw_videos/L21_V001.mp4,media://metadata/L21_V001.json,L21_V001.mp4,.mp4,30.0,ffprobe_avg_frame_rate,1261.633333,1280,720,37849,False,ffprobe_nb_frames,1261.633333,L21_V001
1,L21_V002,media://raw_videos/L21_V002.mp4,media://metadata/L21_V002.json,L21_V002.mp4,.mp4,30.0,ffprobe_avg_frame_rate,1057.333333,1280,720,31720,False,ffprobe_nb_frames,1057.333333,L21_V002
2,L21_V003,media://raw_videos/L21_V003.mp4,media://metadata/L21_V003.json,L21_V003.mp4,.mp4,25.0,ffprobe_avg_frame_rate,1197.840000,1280,720,29946,False,ffprobe_nb_frames,1197.840000,L21_V003
3,L21_V005,media://raw_videos/L21_V005.mp4,media://metadata/L21_V005.json,L21_V005.mp4,.mp4,30.0,ffprobe_avg_frame_rate,943.133333,1280,720,28294,False,ffprobe_nb_frames,943.133333,L21_V005
4,L21_V006,media://raw_videos/L21_V006.mp4,media://metadata/L21_V006.json,L21_V006.mp4,.mp4,30.0,ffprobe_avg_frame_rate,1035.466667,1280,720,31064,False,ffprobe_nb_frames,1035.466667,L21_V006



media_store_manifest.parquet: /content/output/canonical_release_v002/raw_mapping/media_store_manifest.parquet
media_manifest_rows: 834


,video_id,video_ref,metadata_ref,video_filename,metadata_filename,video_size_bytes,metadata_size_bytes,canonical_backend,canonical_repo_id,canonical_repo_type,canonical_revision,canonical_prefix,canonical_video_path,canonical_metadata_path
0,L21_V001,media://raw_videos/L21_V001.mp4,media://metadata/L21_V001.json,L21_V001.mp4,L21_V001.json,130322332,178,hf_dataset,1thesudden/AIC2026_raw,dataset,main,canonical_dataset_v002,raw_videos/L21_V001.mp4,metadata/L21_V001.json
1,L21_V002,media://raw_videos/L21_V002.mp4,media://metadata/L21_V002.json,L21_V002.mp4,L21_V002.json,97471539,178,hf_dataset,1thesudden/AIC2026_raw,dataset,main,canonical_dataset_v002,raw_videos/L21_V002.mp4,metadata/L21_V002.json
2,L21_V003,media://raw_videos/L21_V003.mp4,media://metadata/L21_V003.json,L21_V003.mp4,L21_V003.json,141874706,178,hf_dataset,1thesudden/AIC2026_raw,dataset,main,canonical_dataset_v002,raw_videos/L21_V003.mp4,metadata/L21_V003.json
3,L21_V005,media://raw_videos/L21_V005.mp4,media://metadata/L21_V005.json,L21_V005.mp4,L21_V005.json,83687789,178,hf_dataset,1thesudden/AIC2026_raw,dataset,main,canonical_dataset_v002,raw_videos/L21_V005.mp4,metadata/L21_V005.json
4,L21_V006,media://raw_videos/L21_V006.mp4,media://metadata/L21_V006.json,L21_V006.mp4,L21_V006.json,96238537,178,hf_dataset,1thesudden/AIC2026_raw,dataset,main,canonical_dataset_v002,raw_videos/L21_V006.mp4,metadata/L21_V006.json



batch_manifest.csv: /content/output/canonical_release_v002/manifests/batch_manifest.csv
batch_rows: 834


,batch_id,video_id,estimated_compute_cost,assigned_worker,status,structure_artifact_path,feature_artifact_path,error_note
0,batch_000,L22_V009,1077.080000,worker_000,pending,artifacts/structure/L22_V009_structure.zip,artifacts/features/L22_V009_features.zip,NaN
1,batch_000,L22_V012,1204.766667,worker_000,pending,artifacts/structure/L22_V012_structure.zip,artifacts/features/L22_V012_features.zip,NaN
2,batch_000,L22_V019,1033.966667,worker_000,pending,artifacts/structure/L22_V019_structure.zip,artifacts/features/L22_V019_features.zip,NaN
3,batch_000,L22_V024,1268.200000,worker_000,pending,artifacts/structure/L22_V024_structure.zip,artifacts/features/L22_V024_features.zip,NaN
4,batch_000,L22_V025,1129.166667,worker_000,pending,artifacts/structure/L22_V025_structure.zip,artifacts/features/L22_V025_features.zip,NaN



Batch txt files:
  /content/output/canonical_release_v002/manifests/batch_000.txt
  /content/output/canonical_release_v002/manifests/batch_001.txt
  /content/output/canonical_release_v002/manifests/batch_002.txt
  /content/output/canonical_release_v002/manifests/batch_003.txt
  /content/output/canonical_release_v002/manifests/batch_004.txt
  /content/output/canonical_release_v002/manifests/batch_005.txt
  /content/output/canonical_release_v002/manifests/batch_006.txt
  /content/output/canonical_release_v002/manifests/batch_007.txt
  /content/output/canonical_release_v002/manifests/batch_008.txt
  /content/output/canonical_release_v002/manifests/batch_009.txt

Notebook 00 hoàn tất nếu cell này chạy xong.


In [42]:
# BƯỚC 17 OPTIONAL: Kiểm tra lại HF raw repo versioned sau toàn bộ notebook.
import os

try:
    from huggingface_hub import HfApi
except ImportError:
    %pip install -q huggingface_hub
    from huggingface_hub import HfApi

raw_repo_id = config.hf_canonical_repo
raw_import_id = config.raw_import_id.strip("/")

print("BƯỚC 17 OPTIONAL: Check HF raw repo")
print("- run_upload_standardized_raw:", config.run_upload_standardized_raw)
print("- raw_repo_id:", raw_repo_id)
print("- raw_import_id:", raw_import_id)

if not config.run_upload_standardized_raw:
    print("config.run_upload_standardized_raw=False, skip HF raw repo check.")
else:
    hf_token = os.environ.get("AIC_HF_TOKEN") or os.environ.get("HF_TOKEN")
    if not hf_token:
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
            if hf_token:
                os.environ["HF_TOKEN"] = hf_token
                os.environ["AIC_HF_TOKEN"] = hf_token
        except Exception:
            pass
    if not hf_token:
        raise RuntimeError("Thiếu HF_TOKEN/AIC_HF_TOKEN để kiểm tra HF raw repo.")

    api = HfApi(token=hf_token)
    files = sorted(api.list_repo_files(
        repo_id=raw_repo_id,
        repo_type="dataset",
        token=hf_token,
    ))

    prefix = f"{raw_import_id}/"
    prefix_files = [f for f in files if f.startswith(prefix)]

    raw_files = [f for f in prefix_files if f.startswith(f"{raw_import_id}/raw_videos/")]
    metadata_files = [f for f in prefix_files if f.startswith(f"{raw_import_id}/metadata/")]
    manifest_files = [f for f in prefix_files if f.startswith(f"{raw_import_id}/manifests/")]

    print("prefix_file_count:", len(prefix_files))
    print("raw_videos count:", len(raw_files))
    print("metadata count:", len(metadata_files))
    print("manifests:")
    for f in manifest_files:
        print(" ", f)

    required = [
        f"{raw_import_id}/manifests/canonical_file_manifest.jsonl",
        f"{raw_import_id}/manifests/canonical_import_report.json",
    ]

    missing = [p for p in required if p not in files]

    forbidden = [
        f for f in prefix_files
        if "standardize_progress.jsonl" in f
        or "standardize_archives_report.json" in f
        or "drive_shadow_report.json" in f
        or "batch_" in f
        or f.endswith("videos.parquet")
        or f.endswith("media_store_manifest.parquet")
    ]

    print("missing required:", missing)
    print("forbidden files:", forbidden)

    if not raw_files:
        raise RuntimeError("HF raw repo không có raw_videos.")
    if not metadata_files:
        raise RuntimeError("HF raw repo không có metadata.")
    if missing:
        raise RuntimeError(f"HF raw repo thiếu required manifests: {missing}")
    if forbidden:
        raise RuntimeError(f"HF raw repo có file không đúng mục đích: {forbidden}")

    print("HF raw repo versioned structure OK.")

BƯỚC 11 OPTIONAL: Check HF raw repo
- run_upload_standardized_raw: True
- raw_repo_id: 1thesudden/AIC2026_raw
- raw_import_id: canonical_dataset_v002
prefix_file_count: 1670
raw_videos count: 834
metadata count: 834
manifests:
  canonical_dataset_v002/manifests/canonical_file_manifest.jsonl
  canonical_dataset_v002/manifests/canonical_import_report.json
missing required: []
forbidden files: []
HF raw repo versioned structure OK.
